<a href="https://colab.research.google.com/github/hansheng-hsu/AI-chatbot-for-PDC/blob/main/RAG_with_full_JSON_20240801.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load library

In [ ]:
#connect to google drive
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install langchain weaviate_client wasabi langchain_community langchain_weaviate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.6/990.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.3/368.3 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.8/379.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.1/

In [ ]:
# Install key libraries for LLM

#Install llama-cpp-python with CUBLAS, compatible to CUDA 12.2 which is the CUDA driver build above
!set LLAMA_CUBLAS=1
!set CMAKE_ARGS=-DLLAMA_CUBLAS=on
!set FORCE_CMAKE=1

#Install llama-cpp-python, cuda-enabled package
!python -m pip install llama-cpp-python==0.2.7 --prefer-binary --extra-index-url=https://jllllll.github.io/llama-cpp-python-cuBLAS-wheels/AVX2/cu122

#Install pytorch-related, cuda-enabled package
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://jllllll.github.io/llama-cpp-python-cuBLAS-wheels/AVX2/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 65.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 69.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 44.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 90.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━

In [ ]:
import requests
import json
import weaviate
from weaviate.embedded import EmbeddedOptions
from wasabi import msg
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Fetch the Data

In [ ]:
# Step 1: Fetch the study and protocol data
study_protocol_url = "https://pdc.cancer.gov/graphql"
study_protocol_query = """
query($acceptDUA: Boolean!) {
  study(acceptDUA: $acceptDUA) {
    study_id
    pdc_study_id
    study_submitter_id
    program_id
    project_id
    study_name
    program_name
    project_name
    disease_type
    primary_site
    analytical_fraction
    experiment_type
    embargo_date
    cases_count
    aliquots_count
  }
  protocolPerStudy {
    protocol_id
    protocol_submitter_id
    study_id
    pdc_study_id
    study_submitter_id
    program_id
    program_submitter_id
    protocol_name
    protocol_date
  }
}
"""
variables = {
    "acceptDUA": True
}
response = requests.post(study_protocol_url, json={'query': study_protocol_query, 'variables': variables})
data = response.json()

# Step 2: Merge study and protocol data
merged_data = {}

for study in data['data']['study']:
    pdc_study_id = study['pdc_study_id']
    if pdc_study_id not in merged_data:
        merged_data[pdc_study_id] = study
    else:
        merged_data[pdc_study_id].update(study)

for protocol in data['data']['protocolPerStudy']:
    pdc_study_id = protocol['pdc_study_id']
    if pdc_study_id not in merged_data:
        merged_data[pdc_study_id] = protocol
    else:
        for key, value in protocol.items():
            if key not in merged_data[pdc_study_id]:
                merged_data[pdc_study_id][key] = value

# Step 3: Fetch publication data
publication_url = "https://pdc.cancer.gov/graphql"
publication_query = """
{
  getPaginatedUIPublication(offset: 0, limit: 60) {
    uiPublication {
      publication_id
      pubmed_id
      doi
      title
      journal
      journal_url
      year
      abstract
      studies {
        pdc_study_id
        submitter_id_name
      }
    }
  }
}
"""
pub_response = requests.post(publication_url, json={'query': publication_query})
pub_data = pub_response.json()

# Step 4: Create a mapping of pdc_study_id to publication info
publication_mapping = {}

for publication in pub_data['data']['getPaginatedUIPublication']['uiPublication']:
    for study in publication['studies']:
        pdc_study_id = study['pdc_study_id']
        if pdc_study_id not in publication_mapping:
            publication_mapping[pdc_study_id] = []
        publication_mapping[pdc_study_id].append({
            'publication_id': publication['publication_id'],
            'pubmed_id': publication['pubmed_id'],
            'doi': publication['doi'],
            'title': publication['title'],
            'journal': publication['journal'],
            'journal_url': publication['journal_url'],
            'year': publication['year'],
            'abstract': publication['abstract'],
        })

# Step 5: Add publication info to merged data
for pdc_study_id, publications in publication_mapping.items():
    if pdc_study_id in merged_data:
        merged_data[pdc_study_id]['publications'] = publications
    else:
        merged_data[pdc_study_id] = {'publications': publications}

# Convert merged_data back to a list if needed
merged_list = list(merged_data.values())

# Step 6: Fetch the list of PubMed IDs
publication_query_pubmed_ids = """
{
  getPaginatedUIPublication(offset: 0, limit: 60) {
    total
    uiPublication {
      pubmed_id
    }
  }
}
"""
pub_response_pubmed_ids = requests.post(publication_url, json={'query': publication_query_pubmed_ids})
response_json_pubmed_ids = pub_response_pubmed_ids.json()

# Extract the list of publications
publications = response_json_pubmed_ids['data']['getPaginatedUIPublication']['uiPublication']

# Extract pubmed_id from each publication and create a list
pubmed_ids = [publication['pubmed_id'] for publication in publications]

# Step 7: Fetch article JSON from PubMed API and extract text with sentence splitting
def fetch_article_json(pubmed_id):
    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pubmed_id}/unicode"
    response = requests.get(url)
    response.raise_for_status()  # Check for request errors
    return response.json()

def extract_article_text(article_json):
    article_text = []
    for document in article_json[0]['documents']:
        for passage in document['passages']:
            if passage['infons'].get('section_type') != 'REF':
                article_text.append(passage['text'])
    return " ".join(article_text)

def split_article_text(article_text, chunk_size=256, chunk_overlap=64):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(article_text)

all_articles = []
for pubmed_id in pubmed_ids:
    try:
        article_json = fetch_article_json(pubmed_id)
        article_text = extract_article_text(article_json)
        article_text_split = split_article_text(article_text)
        all_articles.append({ 'pubmed_id': pubmed_id, 'full_text': article_text_split })
    except requests.exceptions.RequestException as e:
        all_articles.append({ 'pubmed_id': pubmed_id, 'full_text': 'Full text is not available in this publication' })
        print(f"Failed to fetch data for PubMed ID {pubmed_id}: {e}")

# Step 8: Merge all_articles with merged_list based on pubmed_id
pubmed_to_full_text = {article['pubmed_id']: article['full_text'] for article in all_articles}

for study in merged_list:
    if 'publications' in study:
        valid_publications = []
        for publication in study['publications']:
            pubmed_id = publication['pubmed_id']
            if pubmed_id in pubmed_to_full_text:
                publication['full_text'] = pubmed_to_full_text[pubmed_id]
                valid_publications.append(publication)
        study['publications'] = valid_publications

# Output the merged data
full_data = {
    study['pdc_study_id']: study for study in merged_list
}
print(json.dumps(full_data, indent=2))


Streaming output truncated to the last 5000 lines.
          "\u03bcl of 50 mM HEPES (pH 8.5). Plate 10 \u03bcl of the diluted sample in triplicate per sample in a clear, flat-bottom 96-well microplate. Plate a 10-\u03bcl HPLC water blank and a 50 mM HEPES, pH 8.5, blank, each in triplicate. Plate eight wells of 10 \u03bcl each of an",
          "each in triplicate. Plate eight wells of 10 \u03bcl each of an 8-point BSA curve, ranging from 50 \u03bcg/ml to 2 mg/ml. Prepare the BCA reagent by mixing reagent B and reagent A in a 1:49 (vol/vol) ratio. \u25b2 CRITICAL STEP This solution must be made fresh",
          "ratio. \u25b2 CRITICAL STEP This solution must be made fresh immediately before use. Add 200 \u03bcl of freshly prepared BCA reagent to each well containing 10 \u03bcl of sample. Incubate the microplate at 37 \u00b0C for 30 min. Read the plate at 562 nm. We usually recover",
          "37 \u00b0C for 30 min. Read the plate at 562 nm. We usually recover 40\u201350% of the star

# Connecting Weaviate

In [ ]:
client = weaviate.Client(
  embedded_options=EmbeddedOptions(),
  additional_headers={
    "X-HuggingFace-Api-Key": "hf_******************"
  }
)

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.26.1/weaviate-v1.26.1-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 2954


# Create the Schema

In [ ]:
from wasabi import msg

# Define a schema for the data
class_obj ={
    "classes": [
        {
            "class": "Document",
            "vectorizer": "text2vec-huggingface",
            "moduleConfig": {
                "text2vec-huggingface": {
                    "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
                    "vectorizeClassName": False
                }
            },
            "properties": [
                {
                    "name": "study_id",
                    "dataType": ["string"],
                    "description": "A unique identifier for a study."
                },
                {
                    "name": "pdc_study_id",
                    "dataType": ["string"],
                    "description": "The PDC-specific study ID (e.g., PDC000121)."
                },
                {
                    "name": "study_submitter_id",
                    "dataType": ["string"],
                    "description": "An identifier submitted by the study’s submitter."
                },
                {
                    "name": "program_id",
                    "dataType": ["string"],
                    "description": "Identifier for the program associated with the study."
                },
                {
                    "name": "project_id",
                    "dataType": ["string"],
                    "description": "Identifier for the project within the program."
                },
                {
                    "name": "study_name",
                    "dataType": ["string"],
                    "description": "Name of the study."
                },
                {
                    "name": "program_name",
                    "dataType": ["string"],
                    "description": "Name of the program."
                },
                {
                    "name": "project_name",
                    "dataType": ["string"],
                    "description": "Name of the project."
                },
                {
                    "name": "disease_type",
                    "dataType": ["string"],
                    "description": "Type of disease being studied."
                },
                {
                    "name": "primary_site",
                    "dataType": ["string"],
                    "description": "Anatomic site of the tumor or disease."
                },
                {
                    "name": "analytical_fraction",
                    "dataType": ["string"],
                    "description": "The type of analytical fraction used (e.g., proteome, phosphoproteome)."
                },
                {
                    "name": "experiment_type",
                    "dataType": ["string"],
                    "description": "Type of labeling reagent used. Indicate number of channels."
                },
                {
                    "name": "embargo_date",
                    "dataType": ["date"],
                    "description": "Date when study data becomes publicly accessible (null if not applicable)."
                },
                {
                    "name": "cases_count",
                    "dataType": ["int"],
                    "description": "Number of cases in the study (null if not available)."
                },
                {
                    "name": "aliquots_count",
                    "dataType": ["int"],
                    "description": "Number of aliquots (samples) in the study (null if not available)."
                },
                {
                    "name": "protocol_id",
                    "dataType": ["string"],
                    "description": "Identifier for the experimental protocol."
                },
                {
                    "name": "protocol_submitter_id",
                    "dataType": ["string"],
                    "description": "Identifier submitted by the protocol submitter."
                },
                {
                    "name": "program_submitter_id",
                    "dataType": ["string"],
                    "description": "Identifier submitted by the program submitter."
                },
                {
                    "name": "protocol_name",
                    "dataType": ["string"],
                    "description": "Name of the experimental protocol."
                },
                {
                    "name": "protocol_date",
                    "dataType": ["date"],
                    "description": "Date protocol was created or updated."
                },
                {
                    "name": "publication_id",
                    "dataType": ["string"],
                    "description": "Publication identifier."
                },
                {
                    "name": "pubmed_id",
                    "dataType": ["string"],
                    "description": "PubMed identifier."
                },
                {
                    "name": "doi",
                    "dataType": ["string"],
                    "description": "Digital Object Identifier (DOI) for the publication."
                },
                {
                    "name": "author",
                    "dataType": ["text"],
                    "description": "Author(s) of the publication."
                },
                {
                    "name": "title",
                    "dataType": ["string"],
                    "description": "Title of the publication."
                },
                {
                    "name": "journal",
                    "dataType": ["string"],
                    "description": "Journal where the publication appeared."
                },
                {
                    "name": "journal_url",
                    "dataType": ["string"],
                    "description": "URL to the journal."
                },
                {
                    "name": "year",
                    "dataType": ["string"],
                    "description": "Year of publication."
                },
                {
                    "name": "abstract",
                    "dataType": ["text"],
                    "description": "Abstract of the publication."
                },
                {
                    "name": "full_text",
                    "dataType": ["text"],
                    "description": "Full text of the publication."
                }
            ]
        }
    ]
}


if not client.schema.exists("Document"):
    client.schema.create(class_obj)
    msg.warn(f"Document class was created because it didn't exist.")
else:
    # WARNING THIS DELETES ALL Document AND CREATES A NEW Document CLASS
    client.schema.delete_class("Document")
    msg.info(f"Document class was removed because it already exists")
    client.schema.create(class_obj)

⚠ Document class was created because it didn't exist.


In [ ]:
def jprint(json_in):
    import json
    print(json.dumps(json_in, indent=2))
jprint(client.schema.get())

{
  "classes": [
    {
      "class": "Document",
      "invertedIndexConfig": {
        "bm25": {
          "b": 0.75,
          "k1": 1.2
        },
        "cleanupIntervalSeconds": 60,
        "stopwords": {
          "additions": null,
          "preset": "en",
          "removals": null
        }
      },
      "moduleConfig": {
        "text2vec-huggingface": {
          "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
          "vectorizeClassName": false
        }
      },
      "multiTenancyConfig": {
        "autoTenantActivation": false,
        "autoTenantCreation": false,
        "enabled": false
      },
      "properties": [
        {
          "dataType": [
            "text"
          ],
          "description": "A unique identifier for a study.",
          "indexFilterable": true,
          "indexRangeFilters": false,
          "indexSearchable": true,
          "moduleConfig": {
            "text2vec-huggingface": {
              "skip": fal

# Import Data to Weaviate

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import logging
import numpy as np
import time

# Embed Text Data using neuml/pubmedbert-base-embeddings
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
model = AutoModel.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")

def generate_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy().tolist()


# Configure logging
logging.basicConfig(level=logging.INFO)
msg = logging.getLogger()

with client.batch as batch:
    batch.batch_size = 100
    for i, (pdc_study_id, doc) in enumerate(full_data.items()):
        msg.info(f"({i+1}/{len(full_data)}) Importing Study {doc.get('pdc_study_id', 'Unknown ID')}")

        # Handle multiple publications if present
        documents = []
        study_info = (
            f"PDC Study ID: {doc['pdc_study_id']}, "
            f"Study Submitter ID: {doc['study_submitter_id']}, "
            f"Program ID: {doc['program_id']}, "
            f"Project ID: {doc['project_id']}, "
            f"Study Name: {doc['study_name']}, "
            f"Program Name: {doc['program_name']}, "
            f"Project Name: {doc['project_name']}, "
            f"Disease Type: {doc['disease_type']}, "
            f"Primary Site: {doc['primary_site']}, "
            f"Analytical Fraction: {doc['analytical_fraction']}, "
            f"Experiment Type: {doc['experiment_type']}, "
            f"Embargo Date: {doc['embargo_date']}, "
            f"Cases Count: {doc['cases_count']}, "
            f"Aliquots Count: {doc['aliquots_count']}, "
            f"Protocol ID: {doc['protocol_id']}, "
            f"Protocol Submitter ID: {doc['protocol_submitter_id']}, "
            f"Program Submitter ID: {doc['program_submitter_id']}, "
            f"Protocol Name: {doc['protocol_name']}, "
            f"Protocol Date: {doc['protocol_date']}"
        )

        # Adding publication information if available
        if 'publications' in doc and doc['publications']:
            for pub in doc['publications']:
                pub_info = (
                    f"Publication ID: {pub['publication_id']}, "
                    f"PubMed ID: {pub['pubmed_id']}, "
                    f"DOI: {pub['doi']}, "
                    f"Title: {pub['title']}, "
                    f"Journal: {pub['journal']}, "
                    f"Journal URL: {pub['journal_url']}, "
                    f"Year: {pub['year']}, "
                    f"Abstract: {pub['abstract']} "
                )
                study_info += f", Publication: ({pub_info})"

        documents.append(study_info)
        embedding = generate_embedding(' '.join(documents))

        properties = {
            "study_id": doc.get("study_id", ""),
            "pdc_study_id": doc.get("pdc_study_id", ""),
            "study_submitter_id": doc.get("study_submitter_id", ""),
            "program_id": doc.get("program_id", ""),
            "project_id": doc.get("project_id", ""),
            "study_name": doc.get("study_name", ""),
            "program_name": doc.get("program_name", ""),
            "project_name": doc.get("project_name", ""),
            "disease_type": doc.get("disease_type", ""),
            "primary_site": doc.get("primary_site", ""),
            "analytical_fraction": doc.get("analytical_fraction", ""),
            "experiment_type": doc.get("experiment_type", ""),
            "cases_count": doc.get("cases_count", None),
            "aliquots_count": doc.get("aliquots_count", None),
            "text_to_embed": ' '.join(documents)
        }

        if 'publications' in doc and doc['publications']:
            for pub in doc['publications']:
                properties.update({
                    "publication_id": pub.get("publication_id", ""),
                    "pubmed_id": pub.get("pubmed_id", ""),
                    "doi": pub.get("doi", ""),
                    "author": pub.get("author", ""),
                    "title": pub.get("title", ""),
                    "journal": pub.get("journal", ""),
                    "journal_url": pub.get("journal_url", ""),
                    "year": pub.get("year", ""),
                    "abstract": pub.get("abstract", "")
                })

        client.batch.add_data_object(properties, "Document", vector=embedding)
        msg.info("Data imported")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

# Check data import

In [ ]:
where_filter= {
    "path":["text_to_embed"],
    "operator":"Like",
    "valueText":"*leukemia*"
}
result1 = (
    client.query
    .get("Document",["pdc_study_id", "study_name", "primary_site"])
    .with_limit(20)
    .with_where(where_filter)
    .do()
)
print(json.dumps(result1, indent=4))


{
    "data": {
        "Get": {
            "Document": [
                {
                    "pdc_study_id": "PDC000478",
                    "primary_site": "Hematopoietic and reticuloendothelial systems;Not Reported",
                    "study_name": "Beat AML Baseline Clinical - Phosphoproteome"
                },
                {
                    "pdc_study_id": "PDC000477",
                    "primary_site": "Hematopoietic and reticuloendothelial systems;Not Reported",
                    "study_name": "Beat AML Baseline Clinical - Proteome"
                },
                {
                    "pdc_study_id": "PDC000318",
                    "primary_site": "Hematopoietic and reticuloendothelial systems;Not Reported",
                    "study_name": "AML Gilteritinib Resistance -  Phosphoproteome"
                },
                {
                    "pdc_study_id": "PDC000403",
                    "primary_site": "Hematopoietic and reticuloendothelial systems;N

In [ ]:
from langchain_community.llms import LlamaCpp
llm = LlamaCpp(
    model_path="/content/drive/MyDrive/Meta-Llama-3-8B-Instruct-Q4_K_M.gguf",
    n_ctx=8192,
    temperature=0.6,
    max_tokens=2048,
    top_p=1
)


In [ ]:
def retrieve_documents(client, query_text, top_k=5):
    query_embedding = generate_embedding(query_text)

    # Define the properties to retrieve
    properties = [
        "study_id", "pdc_study_id", "study_submitter_id", "study_name",
        "program_name", "project_name", "disease_type", "primary_site",
        "analytical_fraction", "experiment_type", "cases_count",
        "aliquots_count", "pubmed_id", "doi", "author", "title",
        "journal", "journal_url", "year", "abstract", "text_to_embed"
    ]

    try:
        results = client.query.get(
            "Document",
            properties=properties
        ).with_near_vector({
            "vector": query_embedding,
            "certainty": 0.7
        }).with_limit(top_k).do()

        response = results['data']['Get']['Document']

        # Create a list of dictionaries with only the desired fields
        return [{'pdc_study_id': doc.get('pdc_study_id'), 'study_name': doc.get('study_name'), 'primary_site': doc.get('primary_site')} for doc in response]

    except Exception as e:
        logging.error(f"An error occurred while retrieving documents: {e}")
        return []

# Example usage
question = "Provide studies related to breast cancer."
retrieve_documents(client, query_text=question)


[{'pdc_study_id': 'PDC000526',
  'study_name': 'CPTAC PDAC Proteins in Serum - Proteome',
  'primary_site': 'Not Applicable;Pancreas'},
 {'pdc_study_id': 'PDC000325',
  'study_name': 'Microscaled Proteogenomic Methods for Precision Oncology DP1 Clinical Trial - Proteome',
  'primary_site': 'Breast;Not Reported'},
 {'pdc_study_id': 'PDC000329',
  'study_name': 'Microscaled Proteogenomic Methods for Precision Oncology PDX cores - Proteome',
  'primary_site': 'Not Reported'},
 {'pdc_study_id': 'PDC000327',
  'study_name': 'Microscaled Proteogenomic Methods for Precision Oncology PDX bulk - Proteome',
  'primary_site': 'Not Reported'},
 {'pdc_study_id': 'PDC000303',
  'study_name': 'Therapeutic Targets in Breast Cancer Xenografts -  Proteome',
  'primary_site': 'Not Reported'}]

# Load llm

In [ ]:
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA, LLMChain

In [ ]:
llm = LlamaCpp(
    model_path= "/content/drive/MyDrive/HIDS-7950/llm/BioMistral-7B.Q4_K_M.gguf",
    n_gpu_layers=-1,
    temperature=0.3,
    max_tokens=2048,
    n_ctx=4096,
    n_batch=512,
    top_p=1)

ValidationError: 1 validation error for LlamaCpp
__root__
  Could not load Llama model from path: /content/drive/MyDrive/HIDS-7950/llm/BioMistral-7B.Q4_K_M.gguf. Received error Model path does not exist: /content/drive/MyDrive/HIDS-7950/llm/BioMistral-7B.Q4_K_M.gguf (type=value_error)

# Set Prompt

In [ ]:
from langchain.chains import LLMChain
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

DEFAULT_LLAMA_SEARCH_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""<<SYS>>
    You are a helpful assistant eager to assist with providing better search results.
    <</SYS>>

    [INST] Provide an answer to the following question. Ensure that the answer is informative, \
            relevant, and concise:
            {question}
    [/INST]""",
)

DEFAULT_SEARCH_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are a helpful assistant eager to assist with providing better search results. \
        Provide an answer to the following question. Ensure that the answer is informative, \
        relevant, and concise: \
        {question}""",
)

QUESTION_PROMPT_SELECTOR = ConditionalPromptSelector(
    default_prompt=DEFAULT_SEARCH_PROMPT,
    conditionals=[(lambda llm: isinstance(llm, LlamaCpp), DEFAULT_LLAMA_SEARCH_PROMPT)],
)

prompt = QUESTION_PROMPT_SELECTOR.get_prompt(llm)
prompt

PromptTemplate(input_variables=['question'], template='<<SYS>>\n    You are a helpful assistant eager to assist with providing better search results.\n    <</SYS>>\n\n    [INST] Provide an answer to the following question. Ensure that the answer is informative,             relevant, and concise:\n            {question}\n    [/INST]')

# Test LLM

In [ ]:
llm_chain = LLMChain(prompt=prompt, llm=llm)
question = "What is Acute Myeloid Leukemia?"
llm_chain.invoke({"question": question})

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(

llama_print_timings:        load time =    3013.80 ms
llama_print_timings:      sample time =      59.24 ms /    93 runs   (    0.64 ms per token,  1569.99 tokens per second)
llama_print_timings: prompt eval time =   44381.35 ms /    83 tokens (  534.72 ms per token,     1.87 tokens per second)
llama_print_timings:        eval time =   57974.08 ms /    92 runs   (  630.15 ms per token,     1.59 tokens per second)
llama_print_timings:       total time =  102521.96 ms /   175 tokens


{'question': 'What is Acute Myeloid Leukemia?',
 'text': ' Acute myeloid leukemia (AML) is a type of cancer that affects the blood cells in the bone marrow . It is characterized by an overproduction of immature white blood cells, which accumulate in the bone marrow and interfere with the production of normal blood cells. The exact cause of AML is unknown, but it is thought to be related to genetic mutations that affect the development of the blood cells.'}

In [ ]:
llm.invoke("What is Acute Myeloid Leukemia?")


llama_print_timings:        load time =    3045.52 ms
llama_print_timings:      sample time =     104.39 ms /   176 runs   (    0.59 ms per token,  1686.05 tokens per second)
llama_print_timings: prompt eval time =    4914.75 ms /    13 tokens (  378.06 ms per token,     2.65 tokens per second)
llama_print_timings:        eval time =  108637.76 ms /   175 runs   (  620.79 ms per token,     1.61 tokens per second)
llama_print_timings:       total time =  113835.58 ms /   188 tokens


' Acute myeloid leukemia (AML) is a type of blood cancer that affects the myeloid lineage. It is characterized by an overproduction of immature myeloid cells, which accumulate in the bone marrow and interfere with normal hematopoiesis. AML can arise de novo or as a secondary malignancy following other types of blood cancer, such as myelodysplastic syndrome (MDS). The exact cause of AML is unknown, but it is thought to involve both genetic and environmental factors. Treatment options for AML include chemotherapy, radiation therapy, stem cell transplantation, and targeted therapies. Despite advances in treatment, the prognosis for patients with AML remains poor, with a 5-year survival rate of less than 20%.'

# Retriever

In [ ]:
from langchain.retrievers import WeaviateHybridSearchRetriever

# Configuration of the retriever
retriever = WeaviateHybridSearchRetriever(
    client=client,
    index_name="Document",
    text_key="text_to_embed",
    attributes=[
        "study_id", "pdc_study_id", "study_submitter_id", "study_name",
        "program_name", "project_name", "disease_type", "primary_site",
        "analytical_fraction", "experiment_type", "cases_count",
        "aliquots_count", "pubmed_id", "doi", "author", "title",
        "journal", "journal_url", "year", "abstract", "text_to_embed"
    ],
    create_schema_if_missing=True,
)


# RAG chain

In [ ]:
from langchain.chains import RetrievalQA
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True,
    return_source_documents = True
)

In [ ]:
query = "What is the disease_type of study PDC000220 ?"
qa.invoke(query)
# correct



> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.


{'query': 'What is the disease_type of study PDC000220 ?',
 'result': ' The disease type of study PDC000220 is Lung Adenocarcinoma;Lung Squamous Cell Carcinoma;Other.',
 'source_documents': [Document(metadata={'abstract': 'Lung cancer in East Asia is characterized by a high percentage of never-smokers, early onset and predominant EGFR mutations. To illuminate the molecular phenotype of this demographically distinct disease, we performed a deep comprehensive proteogenomic study on a prospectively collected cohort in Taiwan, representing early stage, predominantly female, non-smoking lung adenocarcinoma. Integrated genomic, proteomic, and phosphoproteomic analysis delineated the demographically distinct molecular attributes and hallmarks of tumor progression. Mutational signature analysis revealed age- and gender-related mutagenesis mechanisms, characterized by high prevalence of APOBEC mutational signature in younger females and over-representation of environmental carcinogen-like mutat

In [ ]:
query1 = "If you have any breast cancer studies in the PDC database please list the PDC study ID."
qa.invoke(query1)
# correct but not good



> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.


{'query': 'If you have any breast cancer studies in the PDC database please list the PDC study ID.',
 'result': ' We have added the PDC Study IDs for the two breast cancer studies mentioned above in the text. The PDC Study IDs are PDC000303 and PDC000304 for the proteome and phosphoproteome data, respectively.',
 'source_documents': [Document(metadata={'abstract': 'Acute myeloid leukemia is a poor-prognosis cancer commonly stratified by genetic aberrations, but these mutations are often heterogeneous and fail to consistently predict therapeutic response. Here, we combine transcriptomic, proteomic, and phosphoproteomic datasets with ex vivo drug sensitivity data to help understand the underlying pathophysiology of AML beyond mutations. We measure the proteome and phosphoproteome of 210 patients and combine them with genomic and transcriptomic measurements to identify four proteogenomic subtypes that complement existing genetic subtypes. We build a predictor to classify samples into subt

In [ ]:
query2 = "List studies in the PDC that focus on Metabolomics."
qa.invoke(query2)
#



> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.


{'query': 'List studies in the PDC that focus on Metabolomics.',
 'result': ' The following are studies in the PDC that focus on metabolomics: PDC Study ID: PDC000531, Study Submitter ID: Beat AML Baseline Clinical - Metabolome, Program ID: 10251935-5540-11e8-b664-00a098d917f8, Project ID: 7231aab6-7603-499b-a48d-3e3411143e18, Study Name: Beat AML Baseline Clinical - Metabolome, Program Name: Clinical Proteomic Tumor Analysis Consortium, Project Name: Proteogenomic Translational Research Centers (PTRC), Disease Type: Acute Myeloid Leukemia;Myelodysplastic Syndromes;Other;Other Leukemias, Primary Site: Hematopoietic and reticuloendothelial systems;Not Reported, Analytical Fraction: Metabolome, Experiment Type: LC-MS/MS, Embargo Date: None, Cases Count: 211, Aliquots Count: 211, Protocol ID: 036e298e-38a0-4a5b-a05f-1a54d7e299ad, Protocol Submitter ID: Beat AML Baseline Clinical - Metabolome, Program Submitter ID: Clinical Proteomic Tumor Analysis Consortium, Protocol Name: Beat AML Basel

In [ ]:
query3 = "What is the experiment type of PDC000403 ?"
qa.invoke(query3)
# correct



> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.


{'query': 'What is the experiment type of PDC000403 ?',
 'result': ' The experiment type of PDC000403 is TMT11',
 'source_documents': [Document(metadata={'abstract': 'We undertook a comprehensive proteogenomic characterization of 95 prospectively collected endometrial carcinomas, comprising 83 endometrioid and 12 serous tumors. This analysis revealed possible new consequences of perturbations to the p53 and Wnt/Î²-catenin pathways, identified a potential role for circRNAs in the epithelial-mesenchymal transition, and provided new information about proteomic markers of clinical and genomic tumor subgroups, including relationships to known druggable pathways. An extensive genome-wide acetylation survey yielded insights into regulatory mechanisms linking Wnt signaling and histone acetylation. We also characterized aspects of the tumor immune landscape, including immunogenic alterations, neoantigens, common cancer/testis antigens, and the immune microenvironment, all of which can inform im

In [ ]:
query4 = "How many studies are related to leukemia?"
response=qa.invoke(query4)
print("Response:")
print(response["result"])
print("\nRetrieved context:")
print(response["source_documents"])
#



> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.
Response:
 12

Retrieved context:
[Document(metadata={'abstract': 'Acute Myeloid Leukemia (AML) affects 20,000 patients in the US annually with a five-year survival rate of approximately 25%. One reason for the low survival rate is the high prevalence of clonal evolution that gives rise to heterogeneous sub-populations of leukemic cells with diverse mutation spectra, which eventually leads to disease relapse. This genetic heterogeneity drives the activation of complex signaling pathways that is reflected at the protein level. This diversity makes it difficult to treat AML with targeted therapy, requiring custom patient treatment protocols tailored to each individual’s leukemia. Toward this end, the Beat AML research program prospectively collected genomic and transcriptomic data from over 1000 AML patients and carried out ex vivo drug sensitivity assays to identify genomic signatures that could predict patient-specific drug responses. However, there are inherent weak

In [ ]:
query5 = "How many cases and aliquots are there in study PDC000303 ?"
qa.invoke(query5)
# correct



> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.


{'query': 'How many cases and aliquots are there in study PDC000303 ?',
 'result': ' There are 27 cases and 27 aliquots in study PDC000303.',
 'source_documents': [Document(metadata={'abstract': 'Recent advances in mass spectrometry (MS) have enabled extensive analysis of cancer proteomes. Here, we employed quantitative proteomics to profile protein expression across 24 breast cancer patient-derived xenograft (PDX) models. Integrated proteogenomic analysis shows positive correlation between expression measurements from transcriptomic and proteomic analyses; further, gene expression-based intrinsic subtypes are largely re-capitulated using non-stromal protein markers. Proteogenomic analysis also validates a number of predicted genomic targets in multiple receptor tyrosine kinases. However, several protein/phosphoprotein events such as overexpression of AKT proteins and ARAF, BRAF, HSP90AB1 phosphosites are not readily explainable by genomic analysis, suggesting that druggable translatio

In [ ]:
query6 = "Do you have any breast cancer studies in the PDC database?"
response=qa.invoke(query6)
print("Response:")
print(response["result"])
print("\nRetrieved context:")
print(response["source_documents"])

Response:


> Entering new RetrievalQA chain...


Llama.generate: prefix-match hit



> Finished chain.
 Yes, we have several breast cancer studies in the PDC database including TCGA Breast Cancer CompRef Samples - Proteome, TCGA Breast Cancer CompRef Samples - Phosphoproteome and Therapeutic Targets in Breast Cancer Xenografts - Proteome.

Retrieved context:
[Document(metadata={'abstract': 'Somatic mutations have been extensively characterized in breast cancer, but the effects of these genetic alterations on the proteomic landscape remain poorly understood. Here we describe quantitative mass-spectrometry-based proteomic and phosphoproteomic analyses of 105 genomically annotated breast cancers, of which 77 provided high-quality data. Integrated analyses provided insights into the somatic cancer genome including the consequences of chromosomal loss, such as the 5q deletion characteristic of basal-like breast cancer. Interrogation of the 5q trans-effects against the Library of Integrated Network-based Cellular Signatures, connected loss of CETN3 and SKP1 to elevated expr